# Anime review sentiment, checked against the reviewers' own verdicts

Downloads the reviews of the top 10 anime airing on MyAnimeList this month and checks each sentiment method against the reviewers' own verdicts. Run the cells in order; each one can be run again safely.

**For the transformer model, turn on a GPU first:** *Runtime → Change runtime type → T4 GPU*. Without one, the notebook skips that model and runs everything else.

In [1]:
# 1. Get the code (safe to re-run)
import os
os.chdir("/content")
if os.path.isdir("/content/repo/.git"):
    !git -C /content/repo pull -q
else:
    !git clone -q https://github.com/harsh-github007/Anime-Reviews-Sentimental-Analysis.git /content/repo
%cd -q /content/repo
!pip install -q -e ".[model]"
print("Ready in", os.getcwd())

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 5.3 MB/s eta 0:00:00
  Building editable for anime-review-sentiment (pyproject.toml) ... done
Ready in /content/repo


In [2]:
# 2. Download this month's top anime and their reviews (15–30 minutes)
#    Set USE_DRIVE_FILE = True to analyse your own reviews file from Google Drive instead.
%cd -q /content/repo
import os
TOP = 10
USE_DRIVE_FILE = False

if USE_DRIVE_FILE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = "/content/drive/My Drive/Project/Sentiment Analysis of Anime Reviews/Data/Anime Reviews.xlsx"
    if not os.path.exists(DATA):
        from google.colab import files
        print("Not found in Drive. Choose the reviews file (.xlsx or .csv) to upload:")
        up = files.upload()
        os.makedirs("data", exist_ok=True)
        DATA = os.path.join("data", next(iter(up)))
        os.replace(next(iter(up)), DATA)
else:
    DATA = "data/reviews.csv"
    !python -m animesent scrape --top $TOP --out $DATA
print("Using", DATA)

  504 on /top/anime?filter=airing&limit=25; retrying in 10s
  504 on /top/anime?filter=airing&limit=25; retrying in 20s
  Jikan couldn't serve the chart (HTTPError: HTTP Error 504: Gateway Time-out); reading MyAnimeList's chart page instead
Looking down MyAnimeList's airing chart for 10 shows with at least 0 reviews:
  504 on /anime/61316/reviews?page=1&preliminary=true&spoilers=true; retrying in 10s
  504 on /anime/61316/reviews?page=1&preliminary=true&spoilers=true; retrying in 20s
  Jikan couldn't serve reviews (HTTPError: HTTP Error 504: Gateway Time-out); reading MyAnimeList's review pages instead
   1. Re:Zero kara Hajimeru Isekai Seikatsu 4th Season        score 9.11    83 reviews
   2. Steel Ball Run: JoJo no Kimyou na Bouken                score 9.07     2 reviews
   3. Bleach: Sennen Kessen-hen - Kashin-tan                  score 9.03    31 reviews
   4. One Piece                                               score 8.72   100 reviews
   5. Chiikawa                            

In [3]:
# 3. Score every review and check each method against the verdicts
%cd -q /content/repo
import torch
flag = "--transformer" if torch.cuda.is_available() else ""
if not flag:
    print("No GPU: skipping the transformer model. Turn on a GPU to include it.")
!python -m animesent --data "$DATA" --min-reviews 20 $flag

No GPU: skipping the transformer model. Turn on a GPU to include it.
Loaded 261 reviews (0 without a verdict tag, 0 duplicates dropped).
Scored with TextBlob in 2s
Scored with VADER in 3s
Evaluating (cross-validation by show)…
Best method: TextBlob. See results/results.md


In [4]:
# 4. Show the results
%cd -q /content/repo
import os
from IPython.display import Markdown, display
if os.path.exists("results/results.md"):
    display(Markdown(open("results/results.md").read().replace("figures/", "results/figures/")))
else:
    print("No results yet. Fix the message from step 3, then run it again.")

# Results

261 reviews of 9 shows, each tagged by its author as Recommended (175), Mixed Feelings (33), Not Recommended (53).
Dropped before analysis: 0 with no verdict tag, 0 empty, 0 duplicates.

## The anime

The top 10 anime on MyAnimeList's top airing chart on 2026-09-30, and each one's newest 100 reviews.

| Chart rank | Anime | MAL score | Reviews |
| ---: | --- | ---: | ---: |
| 1 | Re:Zero kara Hajimeru Isekai Seikatsu 4th Season | 9.11 | 83 |
| 2 | Steel Ball Run: JoJo no Kimyou na Bouken | 9.07 | 2 |
| 3 | Bleach: Sennen Kessen-hen - Kashin-tan | 9.03 | 31 |
| 4 | One Piece | 8.72 | 100 |
| 5 | Chiikawa | 8.62 | 8 |
| 6 | Seihantai na Kimi to Boku 2nd Season | 8.51 | 18 |
| 7 | Shiguang Dailiren III | 8.51 | 2 |
| 8 | Xian Ni | 8.49 | 13 |
| 9 | Tian Guan Cifu Short Films | 8.47 | 0 |
| 10 | Doupo Cangqiong: Nian Fan | 8.39 | 4 |

## Latest reviews vs the MyAnimeList score

The score MyAnimeList shows is the average of every user's rating. Each review also carries its author's 1–10 rating, so the latest reviews can be checked against it.

Across the 9 shows with rated reviews, the latest reviews average **0.79 points below** the site score, and are **0.99 points away** on average. 3 of 9 land within half a point.

| Chart rank | Anime | MAL score | Latest reviews | Their average rating | Gap | Recommended | Reviews dated |
| ---: | --- | ---: | ---: | ---: | ---: | ---: | --- |
| 1 | Re:Zero kara Hajimeru Isekai Seikatsu 4th Season | 9.11 | 83 | 7.81 | -1.30 | 66% | 2026-04-22 to 2026-09-25 |
| 2 | Steel Ball Run: JoJo no Kimyou na Bouken | 9.07 | 2 | 10.00 | +0.93 | 100% | 2026-09-28 to 2026-09-28 |
| 3 | Bleach: Sennen Kessen-hen - Kashin-tan | 9.03 | 31 | 7.71 | -1.32 | 71% | 2026-08-12 to 2026-09-23 |
| 4 | One Piece | 8.72 | 100 | 7.62 | -1.10 | 60% | 2025-04-02 to 2026-09-12 |
| 5 | Chiikawa | 8.62 | 8 | 8.38 | -0.24 | 100% | 2023-10-29 to 2026-09-19 |
| 6 | Seihantai na Kimi to Boku 2nd Season | 8.51 | 18 | 8.44 | -0.07 | 89% | 2026-07-21 to 2026-09-29 |
| 7 | Shiguang Dailiren III | 8.51 | 2 | 6.50 | -2.01 | 50% | 2026-08-31 to 2026-09-27 |
| 8 | Xian Ni | 8.49 | 13 | 6.92 | -1.57 | 62% | 2024-06-15 to 2026-09-14 |
| 9 | Tian Guan Cifu Short Films | 8.47 | 0 | – | – | – | no reviews |
| 10 | Doupo Cangqiong: Nian Fan | 8.39 | 4 | 8.00 | -0.39 | 75% | 2023-08-03 to 2025-05-11 |

Reviews are a small, self-selected group: people who write a review often feel strongly, and a show's latest reviews reflect its latest episodes, while the site score covers everyone who rated it.

## What TextBlob says

TextBlob calls 90.8% of reviews positive. The reviewers themselves recommend 67.0% of the shows they review, and TextBlob scores 75% of **Not Recommended** reviews as positive.

![TextBlob polarity by verdict](results/figures/textblob_by_verdict.png)

## How well each method matches the verdict

Recommended vs Not Recommended: 228 reviews (175 vs 53). Five-fold cross-validation split by show, so every review is scored by a model that never saw that show. Balanced accuracy averages the hit rate on each verdict, so always answering "Recommended" scores 50%.

| Method | ROC AUC (95% CI) | Balanced accuracy, usual cut-off | Balanced accuracy, tuned cut-off | Not Recommended caught |
| --- | ---: | ---: | ---: | ---: |
| TextBlob | 0.87 (0.83–0.94) | 61% | 80% | 25% |
| VADER | 0.81 (0.72–0.89) | 71% | 69% | 51% |
| Trained on verdicts | 0.86 (0.77–0.93) | 69% | 76% | 40% |

Best at separating the verdicts: **TextBlob**.

![Methods](results/figures/methods.png)

## Three verdicts

Trained on all three tags, the model reaches a macro F1 of 0.47 and balanced accuracy of 47%. Rows are the reviewer's tag, columns the model's guess:

| | Recommended | Mixed Feelings | Not Recommended |
| --- | ---: | ---: | ---: |
| **Recommended** | 170 | 1 | 4 |
| **Mixed Feelings** | 26 | 1 | 6 |
| **Not Recommended** | 32 | 0 | 21 |

## By show

Shows with at least 20 reviews (3).

![Shows](results/figures/titles.png)

| Show | Reviews | Recommended (95% CI) | Not Recommended | TextBlob positive |
| --- | ---: | ---: | ---: | ---: |
| Bleach: Sennen Kessen-hen - Kashin-tan | 31 | 71% (53%–84%) | 23% | 94% |
| Re:Zero kara Hajimeru Isekai Seikatsu 4th Season | 83 | 66% (56%–76%) | 23% | 88% |
| One Piece | 100 | 60% (50%–69%) | 24% | 91% |

## What the trained model listens to

Words and phrases with the largest weights toward each verdict:

- **Recommended:** best, the best, 10 10, amazing, cour, you, peak, the animation, perfect, very, such, animation, love, especially, world
- **Not Recommended:** plot, same, the worst, worst, bad, the same, the plot, minutes, over, fight, out of, boring, no, barely, rather

## Caveats

- Reviews of the top airing chart on MyAnimeList, downloaded 2026-09-30. Reviewers who write reviews are not all viewers, so these rates describe reviews, not audiences.
- A tag is a single summary of a long, often mixed review; some disagreement with any method is expected.
- Scores come from English text only; reviews in other languages were left as they are.


In [5]:
# 5. Download the results to add to the repository (charts and numbers only, no review text)
%cd -q /content/repo
!zip -qr results.zip results
from google.colab import files
files.download("results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>